In [2]:
import pandas as pd
import numpy as np

# --- 1. Cargar los datos ---
# Asumo que df_asociacion ya está cargado. Si no, puedes cargarlo con esta línea:
try:
    df_asociacion = pd.read_excel('validacion/analisis-asociacion-estaciones.xlsx')
except FileNotFoundError:
    print("Advertencia: No se encontró el archivo 'analisis-asociacion-estaciones.xlsx'.")
    print("Se usará un DataFrame de ejemplo para la demostración.")
    # DataFrame de ejemplo si el archivo no existe
    data = {'Estacion': range(226), 'Correlacion': np.random.rand(226) * 1.1 - 0.1}
    df_asociacion = pd.DataFrame(data)

# --- 2. Definir las condiciones para cada intervalo ---
# Se crea un diccionario para mantener el análisis organizado.
condiciones = {
    'Exactamente 0': (df_asociacion['Correlacion'] == 0),
    '> 0.1 a 0.3': (df_asociacion['Correlacion'] > 0.1) & (df_asociacion['Correlacion'] <= 0.3),
    '> 0.3 a 0.5': (df_asociacion['Correlacion'] > 0.3) & (df_asociacion['Correlacion'] <= 0.5),
    '> 0.5 a 0.8': (df_asociacion['Correlacion'] > 0.5) & (df_asociacion['Correlacion'] <= 0.8),
    '> 0.8 a 1.0': (df_asociacion['Correlacion'] > 0.8) & (df_asociacion['Correlacion'] <= 1.0)
}

# --- 3. Calcular la frecuencia absoluta para cada intervalo ---
frecuencias_abs = {intervalo: cond.sum() for intervalo, cond in condiciones.items()}

# --- 4. Identificar y contar los datos no incluidos en los intervalos anteriores ---
# Para un análisis completo, contamos los valores que no cayeron en tus categorías.
condiciones_combinadas = pd.concat(condiciones.values(), axis=1).any(axis=1)
df_otros = df_asociacion[~condiciones_combinadas]

# Desglosamos la categoría "Otros" para mayor claridad
if not df_otros.empty:
    frecuencias_abs['> 0 a 0.1'] = ((df_otros['Correlacion'] > 0) & (df_otros['Correlacion'] <= 0.1)).sum()
    frecuencias_abs['Negativos (< 0)'] = (df_otros['Correlacion'] < 0).sum()

# --- 5. Crear la tabla de resultados y ordenar los intervalos ---
df_resultados = pd.DataFrame(list(frecuencias_abs.items()), columns=['Intervalo de Correlación', 'Frecuencia Absoluta'])

# Ordenamos los intervalos de forma lógica
orden_intervalos = [
    'Negativos (< 0)', 'Exactamente 0', '> 0 a 0.1', '> 0.1 a 0.3', 
    '> 0.3 a 0.5', '> 0.5 a 0.8', '> 0.8 a 1.0'
]
# Filtramos para mantener solo las categorías que existen en los datos
orden_existente = [cat for cat in orden_intervalos if cat in df_resultados['Intervalo de Correlación'].values]
df_resultados['Intervalo de Correlación'] = pd.Categorical(df_resultados['Intervalo de Correlación'], categories=orden_existente, ordered=True)
df_resultados = df_resultados.sort_values('Intervalo de Correlación').reset_index(drop=True)


# --- 6. Calcular la frecuencia relativa ---
total_datos = len(df_asociacion)
df_resultados['Frecuencia Relativa (%)'] = (df_resultados['Frecuencia Absoluta'] / total_datos * 100).round(2)

# --- 7. Añadir una fila con el total para verificación ---
total_absoluta = df_resultados['Frecuencia Absoluta'].sum()
total_relativa = df_resultados['Frecuencia Relativa (%)'].sum()
fila_total = pd.DataFrame({
    'Intervalo de Correlación': ['Total'],
    'Frecuencia Absoluta': [total_absoluta],
    'Frecuencia Relativa (%)': [total_relativa]
})
df_resultados = pd.concat([df_resultados, fila_total], ignore_index=True)


# --- 8. Mostrar la tabla final ---
print("Tabla de Frecuencias para la columna 'Correlacion':")
print(df_resultados.to_string(index=False))


Tabla de Frecuencias para la columna 'Correlacion':
Intervalo de Correlación  Frecuencia Absoluta  Frecuencia Relativa (%)
           Exactamente 0                    0                      0.0
             > 0.1 a 0.3                    0                      0.0
             > 0.3 a 0.5                    0                      0.0
             > 0.5 a 0.8                   33                     14.6
             > 0.8 a 1.0                  193                     85.4
                   Total                  226                    100.0


In [3]:
import pandas as pd
import numpy as np

# --- 1. Cargar los datos ---
# Asumo que df_asociacion ya está cargado. Si no, puedes cargarlo con esta línea:
try:
    df_asociacion = pd.read_excel('validacion/analisis-asociacion-estaciones.xlsx')
except FileNotFoundError:
    print("Advertencia: No se encontró el archivo 'analisis-asociacion-estaciones.xlsx'.")
    print("Se usará un DataFrame de ejemplo para la demostración.")
    # DataFrame de ejemplo si el archivo no existe
    data = {'Estacion': range(226), 'R2': np.random.rand(226)}
    df_asociacion = pd.DataFrame(data)

# --- 2. Definir las condiciones para cada intervalo de R² ---
# R² siempre está entre 0 y 1, por lo que no necesitamos rangos negativos.
condiciones = {
    'Exactamente 0': (df_asociacion['R2'] == 0),
    '> 0 a 0.1': (df_asociacion['R2'] > 0) & (df_asociacion['R2'] <= 0.1),
    '> 0.1 a 0.3': (df_asociacion['R2'] > 0.1) & (df_asociacion['R2'] <= 0.3),
    '> 0.3 a 0.5': (df_asociacion['R2'] > 0.3) & (df_asociacion['R2'] <= 0.5),
    '> 0.5 a 0.8': (df_asociacion['R2'] > 0.5) & (df_asociacion['R2'] <= 0.8),
    '> 0.8 a 1.0': (df_asociacion['R2'] > 0.8) & (df_asociacion['R2'] <= 1.0)
}

# --- 3. Calcular la frecuencia absoluta para cada intervalo ---
frecuencias_abs = {intervalo: cond.sum() for intervalo, cond in condiciones.items()}

# --- 4. Crear la tabla de resultados y ordenar los intervalos ---
df_resultados = pd.DataFrame(list(frecuencias_abs.items()), columns=['Intervalo de R²', 'Frecuencia Absoluta'])

# Ordenamos los intervalos de forma lógica
orden_intervalos = [
    'Exactamente 0', '> 0 a 0.1', '> 0.1 a 0.3', 
    '> 0.3 a 0.5', '> 0.5 a 0.8', '> 0.8 a 1.0'
]
df_resultados['Intervalo de R²'] = pd.Categorical(df_resultados['Intervalo de R²'], categories=orden_intervalos, ordered=True)
df_resultados = df_resultados.sort_values('Intervalo de R²').reset_index(drop=True)

# --- 5. Calcular la frecuencia relativa ---
total_datos = len(df_asociacion)
df_resultados['Frecuencia Relativa (%)'] = (df_resultados['Frecuencia Absoluta'] / total_datos * 100).round(2)

# --- 6. Añadir una fila con el total para verificación ---
total_absoluta = df_resultados['Frecuencia Absoluta'].sum()
total_relativa = df_resultados['Frecuencia Relativa (%)'].sum()
fila_total = pd.DataFrame({
    'Intervalo de R²': ['Total'],
    'Frecuencia Absoluta': [total_absoluta],
    'Frecuencia Relativa (%)': [total_relativa]
})
df_resultados = pd.concat([df_resultados, fila_total], ignore_index=True)

# --- 7. Mostrar la tabla final ---
print("Tabla de Frecuencias para la columna 'R2':")
print(df_resultados.to_string(index=False))


Tabla de Frecuencias para la columna 'R2':
Intervalo de R²  Frecuencia Absoluta  Frecuencia Relativa (%)
  Exactamente 0                    0                     0.00
      > 0 a 0.1                    0                     0.00
    > 0.1 a 0.3                    0                     0.00
    > 0.3 a 0.5                    7                     3.10
    > 0.5 a 0.8                  163                    72.12
    > 0.8 a 1.0                   56                    24.78
          Total                  226                   100.00


In [4]:
import pandas as pd
import numpy as np

# --- 1. Cargar el archivo Excel ---
# Es una buena práctica manejar la posibilidad de que el archivo no exista.
try:
    df_error = pd.read_excel('validacion/analisis-error.xlsx')
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo 'validacion/analisis-error.xlsx'.")
    print("Asegúrate de haber ejecutado los notebooks anteriores para generar este archivo.")
    # Si el archivo no existe, creamos un DataFrame de ejemplo para no detener el flujo.
    df_error = pd.DataFrame({'Bias': np.random.uniform(-100, 100, 226)})


# --- 2. Crear 4 categorías de igual tamaño desde el mínimo al máximo ---
# pd.cut es la función perfecta para esto. Divide los datos en 4 "bins" o intervalos.
# El resultado es una nueva columna que indica a qué categoría pertenece cada fila.
df_error['Categoria_Bias'] = pd.cut(df_error['Bias'], bins=4)


# --- 3. Calcular la frecuencia absoluta ---
# value_counts() cuenta cuántas veces aparece cada categoría.
# sort_index() ordena los intervalos de menor a mayor.
frecuencia_abs = df_error['Categoria_Bias'].value_counts().sort_index()


# --- 4. Crear la tabla de resultados ---
# Creamos un nuevo DataFrame para mostrar los resultados de forma clara.
df_resultados = pd.DataFrame({
    'Intervalo de Bias': frecuencia_abs.index.astype(str), # Convertimos los intervalos a texto para mejor visualización
    'Frecuencia Absoluta': frecuencia_abs.values
})


# --- 5. Calcular la frecuencia relativa ---
total_datos = len(df_error)
df_resultados['Frecuencia Relativa (%)'] = (df_resultados['Frecuencia Absoluta'] / total_datos * 100).round(2)


# --- 6. Añadir una fila con el total para verificación ---
total_absoluta = df_resultados['Frecuencia Absoluta'].sum()
total_relativa = df_resultados['Frecuencia Relativa (%)'].sum()
fila_total = pd.DataFrame({
    'Intervalo de Bias': ['Total'],
    'Frecuencia Absoluta': [total_absoluta],
    'Frecuencia Relativa (%)': [total_relativa]
})
df_resultados = pd.concat([df_resultados, fila_total], ignore_index=True)


# --- 7. Mostrar la tabla final ---
print("Tabla de Frecuencias para la columna 'Bias':")
print(df_resultados.to_string(index=False))



Tabla de Frecuencias para la columna 'Bias':
  Intervalo de Bias  Frecuencia Absoluta  Frecuencia Relativa (%)
(-319.62, -217.897]                    5                     2.21
(-217.897, -116.58]                   11                     4.87
 (-116.58, -15.263]                   51                    22.57
  (-15.263, 86.054]                  159                    70.35
              Total                  226                   100.00


In [1]:
import pandas as pd
import numpy as np

# --- 1. Cargar el archivo Excel ---
# Es una buena práctica manejar la posibilidad de que el archivo no exista.
try:
    df_error = pd.read_excel('validacion/analisis-error.xlsx')
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo 'validacion/analisis-error.xlsx'.")
    print("Asegúrate de haber ejecutado los notebooks anteriores para generar este archivo.")
    # Si el archivo no existe, creamos un DataFrame de ejemplo para no detener el flujo.
    # Como RMSE no puede ser negativo, generamos valores positivos.
    df_error = pd.DataFrame({'RMSE': np.random.uniform(50, 400, 226)})


# --- 2. Crear 4 categorías de igual tamaño para RMSE ---
# pd.cut divide los datos en 4 intervalos de igual rango.
# El resultado es una nueva columna que indica a qué categoría pertenece cada fila.
df_error['Categoria_RMSE'] = pd.cut(df_error['RMSE'], bins=4)


# --- 3. Calcular la frecuencia absoluta ---
# value_counts() cuenta cuántas veces aparece cada categoría.
# sort_index() ordena los intervalos de menor a mayor.
frecuencia_abs = df_error['Categoria_RMSE'].value_counts().sort_index()


# --- 4. Crear la tabla de resultados ---
# Creamos un nuevo DataFrame para mostrar los resultados de forma clara.
df_resultados = pd.DataFrame({
    'Intervalo de RMSE': frecuencia_abs.index.astype(str), # Convertimos los intervalos a texto
    'Frecuencia Absoluta': frecuencia_abs.values
})


# --- 5. Calcular la frecuencia relativa ---
total_datos = len(df_error)
df_resultados['Frecuencia Relativa (%)'] = (df_resultados['Frecuencia Absoluta'] / total_datos * 100).round(2)


# --- 6. Añadir una fila con el total para verificación ---
total_absoluta = df_resultados['Frecuencia Absoluta'].sum()
total_relativa = df_resultados['Frecuencia Relativa (%)'].sum()
fila_total = pd.DataFrame({
    'Intervalo de RMSE': ['Total'],
    'Frecuencia Absoluta': [total_absoluta],
    'Frecuencia Relativa (%)': [total_relativa]
})
df_resultados = pd.concat([df_resultados, fila_total], ignore_index=True)


# --- 7. Mostrar la tabla final ---
print("Tabla de Frecuencias para la columna 'RMSE':")
print(df_resultados.to_string(index=False))


Tabla de Frecuencias para la columna 'RMSE':
 Intervalo de RMSE  Frecuencia Absoluta  Frecuencia Relativa (%)
 (22.064, 109.024]                  187                    82.74
(109.024, 195.637]                   27                    11.95
 (195.637, 282.25]                    8                     3.54
 (282.25, 368.863]                    4                     1.77
             Total                  226                   100.00


In [2]:
import pandas as pd
import numpy as np

# --- 1. Cargar el archivo Excel ---
# Es una buena práctica manejar la posibilidad de que el archivo no exista.
try:
    # Según el README, el archivo 'analisis-error.xlsx' contiene la columna MAD.
    df_error = pd.read_excel('validacion/analisis-error.xlsx')
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo 'validacion/analisis-error.xlsx'.")
    print("Asegúrate de haber ejecutado los notebooks anteriores para generar este archivo.")
    # Si el archivo no existe, creamos un DataFrame de ejemplo para no detener el flujo.
    # Como MAD no puede ser negativo, generamos valores positivos.
    df_error = pd.DataFrame({'MAD': np.random.uniform(30, 300, 226)})


# --- 2. Crear 4 categorías de igual tamaño para MAD ---
# pd.cut divide los datos en 4 intervalos de igual rango.
# El resultado es una nueva columna que indica a qué categoría pertenece cada fila.
df_error['Categoria_MAD'] = pd.cut(df_error['MAD'], bins=4)


# --- 3. Calcular la frecuencia absoluta ---
# value_counts() cuenta cuántas veces aparece cada categoría.
# sort_index() ordena los intervalos de menor a mayor.
frecuencia_abs = df_error['Categoria_MAD'].value_counts().sort_index()


# --- 4. Crear la tabla de resultados ---
# Creamos un nuevo DataFrame para mostrar los resultados de forma clara.
df_resultados = pd.DataFrame({
    'Intervalo de MAD': frecuencia_abs.index.astype(str), # Convertimos los intervalos a texto
    'Frecuencia Absoluta': frecuencia_abs.values
})


# --- 5. Calcular la frecuencia relativa ---
total_datos = len(df_error)
df_resultados['Frecuencia Relativa (%)'] = (df_resultados['Frecuencia Absoluta'] / total_datos * 100).round(2)


# --- 6. Añadir una fila con el total para verificación ---
total_absoluta = df_resultados['Frecuencia Absoluta'].sum()
total_relativa = df_resultados['Frecuencia Relativa (%)'].sum()
fila_total = pd.DataFrame({
    'Intervalo de MAD': ['Total'],
    'Frecuencia Absoluta': [total_absoluta],
    'Frecuencia Relativa (%)': [total_relativa]
})
df_resultados = pd.concat([df_resultados, fila_total], ignore_index=True)


# --- 7. Mostrar la tabla final ---
print("Tabla de Frecuencias para la columna 'MAD':")
print(df_resultados.to_string(index=False))


Tabla de Frecuencias para la columna 'MAD':
  Intervalo de MAD  Frecuencia Absoluta  Frecuencia Relativa (%)
  (16.758, 92.685]                  199                    88.05
 (92.685, 168.311]                   19                     8.41
(168.311, 243.936]                    5                     2.21
(243.936, 319.561]                    3                     1.33
             Total                  226                   100.00
